# Nova — generate 15 original documentary images with Z-Image Turbo

Run this notebook on Kaggle with **Settings → Accelerator → GPU** and **Internet = On**. It downloads the open model `Tongyi-MAI/Z-Image-Turbo` (no Pollinations key and no paid image API), creates one original image for each scene, then packages them as `nova_ai_frames.zip`.

The free GPU is quota-limited and may be queued. The first run downloads a large model. This notebook does not upload anything or spend API credits.

In [ ]:
!git clone --depth 1 https://github.com/nova-api-del/nova.git /kaggle/working/nova
%cd /kaggle/working/nova
!pip -q install -U git+https://github.com/huggingface/diffusers transformers accelerate sentencepiece protobuf edge-tts requests pillow

In [ ]:
import os, sys, json, zipfile, torch
from pathlib import Path
from diffusers import ZImagePipeline
sys.path.insert(0, '/kaggle/working/nova')
from scripts.longform_image_video import SCENES

assert torch.cuda.is_available(), 'Turn on a GPU in Kaggle Notebook settings, then restart the session.'
props = torch.cuda.get_device_properties(0)
major, minor = torch.cuda.get_device_capability(0)
dtype = torch.bfloat16 if major >= 8 else torch.float16
print('GPU:', props.name, '| VRAM:', round(props.total_memory/1024**3, 1), 'GB | dtype:', dtype)

OUT = Path('/kaggle/working/nova_ai_frames')
OUT.mkdir(parents=True, exist_ok=True)
pipe = ZImagePipeline.from_pretrained(
    'Tongyi-MAI/Z-Image-Turbo',
    torch_dtype=dtype,
    low_cpu_mem_usage=False,
)
pipe.enable_model_cpu_offload()
print('Z-Image Turbo loaded. Starting 15 scenes...')

In [ ]:
records=[]
for i, (title, paragraph, visual) in enumerate(SCENES, 1):
    prompt = (
      'Photorealistic premium natural-history documentary still, cinematic IMAX underwater photography, '
      'wide 16:9 composition, extraordinary detail, physically realistic marine biology, dramatic but natural '
      'volumetric lighting, rich deep-ocean blues, believable scale, professional BBC Earth cinematography. '
      'No text, no labels, no watermark. Scene: ' + visual + '.'
    )
    seed = 26000 + i
    image = pipe(
      prompt=prompt, height=576, width=1024, num_inference_steps=9, guidance_scale=0.0,
      generator=torch.Generator(device='cpu').manual_seed(seed)
    ).images[0]
    path = OUT / f'scene_{i:02d}.png'
    image.save(path)
    records.append({'scene':i,'title':title,'model':'Tongyi-MAI/Z-Image-Turbo','seed':seed,'prompt':prompt,'file':path.name})
    print(f'[{i:02d}/15] Generated: {title}')
    del image
    if torch.cuda.is_available(): torch.cuda.empty_cache()

(OUT/'manifest.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
zip_path='/kaggle/working/nova_ai_frames.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()): z.write(p, arcname=p.name)
print('READY:', zip_path, '| images:', len(records))

In [ ]:
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/nova_ai_frames.zip'))